In [8]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import spearmanr
import joblib

In [9]:
df = pd.read_csv("datasets/final_ds.csv")
target_cols = [f"NCAR{i}" for i in range(1,21)]
subset = target_cols + ['stage', 'date']
df_clean = df.dropna(subset=subset).reset_index(drop=True)

In [10]:
df_clean['date'] = pd.to_datetime(df_clean['date'])

In [11]:
train_mask = (df_clean['date'] >= '2025-01-01') & (df_clean['date'] <= '2025-11-30')
test_mask = (df_clean['date'] >= '2026-01-01') & (df_clean['date'] <= '2026-06-30')

In [12]:
models = {}
eval_metrics = []

for dayX in range(1,21):
    target = f'NCAR{dayX}'
    features = ['stage', 
                'polarity',
                'drug_count',
                'Total Revenue', 
                'Operating Revenue', 
                'Cash And Cash Equivalents', 
                'Capital Expenditure', 
                'Stock_Trend_30d', 
                'Stock_Trend_60d',
                'NBI_Trend_30d', 
                'NBI_Trend_60d',
                'isDelisted'
                ]


    X_train = df_clean.loc[train_mask, features]
    y_train = df_clean.loc[train_mask, target]

    X_test = df_clean.loc[test_mask, features]
    y_test = df_clean.loc[test_mask, target]

    X_train['stage'] = X_train['stage'].astype('category')
    X_test['stage'] = X_test['stage'].astype('category')

    X_train['isDelisted'] = X_train['isDelisted'].astype('int')
    X_test['isDelisted'] = X_test['isDelisted'].astype('int')

    cont_vars = [c for c in features if (c != 'stage' and c!= 'isDelisted')]
    X_train[cont_vars] = X_train[cont_vars].apply(pd.to_numeric)
    X_test[cont_vars] = X_test[cont_vars].apply(pd.to_numeric)

    y_train = y_train.apply(pd.to_numeric)
    y_test = y_test.apply(pd.to_numeric)

    print(f"Train observation count: {len(X_train)}")
    print(f"Test observation count: {len(X_test)}")

    model = xgb.XGBRegressor(
        objective = 'reg:pseudohubererror',
        huber_slope = 1.0,
        eval_metric = 'mae',
        n_estimators = 100,
        learning_rate = 0.05,
        max_depth=5,
        random_state = 42,
        n_jobs = 1,
        tree_method = "hist",
        enable_categorical = True,
        missing = np.nan
    )


    model.fit(X_train,
            y_train,
            eval_set = [(X_test, y_test)],
            verbose= False
            )

    model.save_model("model.json")

    models[dayX] = model

    y_pred = model.predict(X_test)

    # model evaluation

    rank_ic, p_val = spearmanr(y_test, y_pred)

    eval_metrics.append({
        "Day": dayX,
        "Directional Accuracy": np.mean(np.sign(y_pred) == np.sign(y_test)),
        "Rank IC": rank_ic, 
        "p_val": p_val,
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "MAE": mean_absolute_error(y_test, y_pred),
        "R2": r2_score(y_test, y_pred)
    })

    importance = pd.DataFrame({
        'Features' : features,
        'Importance': models[dayX].feature_importances_
    }).sort_values('Importance', ascending = False)

    print(importance)

    labels = range(1,11)

    deciles = pd.qcut(y_pred, q=10, labels = labels)
    spread = y_test.groupby(deciles).mean()

    print(spread)

KeyError: 'isDelisted'

In [ ]:
eval_metrics = pd.DataFrame(eval_metrics)
print(eval_metrics)

    Day  Directional Accuracy   Rank IC         p_val      RMSE       MAE  \
0     1              0.494245  0.083894  1.250285e-03  0.359372  0.076641   
1     2              0.491537  0.090152  5.226967e-04  1.044011  0.115993   
2     3              0.516588  0.121931  2.606795e-06  0.806624  0.136257   
3     4              0.516588  0.121123  3.037926e-06  0.746420  0.146724   
4     5              0.514557  0.138744  8.623727e-08  0.750434  0.163636   
5     6              0.547055  0.209152  4.613143e-16  0.745435  0.171172   
6     7              0.535545  0.168167  7.827920e-11  0.559237  0.171601   
7     8              0.526743  0.170299  4.472316e-11  0.723257  0.193497   
8     9              0.495599  0.125601  1.284210e-06  1.038208  0.216185   
9    10              0.505078  0.137263  1.184519e-07  0.976390  0.213221   
10   11              0.526066  0.199042  1.159338e-14  1.014445  0.220060   
11   12              0.524035  0.162966  2.978160e-10  0.587566  0.210415   

In [ ]:
joblib.dump(models, "xgb_models_1_to_20.joblib")

['xgb_models_1_to_20.joblib']